# RiskSequencer — 03 Attention Visualization

**Explainability is a hard requirement.** For every flagged sequence the model
returns per-timestep attention weights; this notebook turns those into an audit
trail — *which past transactions drove the fraud flag*.

We train a quick model, score the held-out test set, then for the
highest-confidence fraud predictions we plot:
1. the attention weight over the 50-step window, and
2. the trajectories of the three monitored behavioral features,

so you can see attention concentrate around the account-takeover burst.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import torch
import matplotlib.pyplot as plt

from config import FEATURE_COLUMNS, SEQUENCE_LENGTH, MONITORED_FEATURES, TrainConfig
from data.synthetic import generate_transactions
from features.feature_pipeline import build_features
from data.sequence_builder import build_sequences, fit_scaler, time_based_split
from training.train import run_training, _scores

## 1. Build data and train a quick model

In [ ]:
raw = generate_transactions(n_users=2500, seed=42)
feats = build_features(raw)
tr, va, te = time_based_split(feats)
scaler = fit_scaler(tr)
train_ds = build_sequences(tr, scaler)
val_ds   = build_sequences(va, scaler)
test_ds  = build_sequences(te, scaler)

out = run_training(train_ds, val_ds, TrainConfig(max_epochs=12))
model = out.model.eval()
print("validation AUC:", round(out.val_result.auc_roc, 4))

## 2. Score the test set and extract attention

`forward` returns `(logits, attn_weights)` with shape `(B, T, 1)`. Padding positions are masked to ~0 weight.

In [ ]:
with torch.no_grad():
    x = torch.from_numpy(test_ds.X)
    m = torch.from_numpy(test_ds.mask)
    logits, attn = model(x, m)              # attn: (N, T, 1)
    probs = torch.sigmoid(logits).squeeze(-1).numpy()
attn = attn.squeeze(-1).numpy()             # (N, T)

# highest-confidence predicted-fraud sequences that are actually fraud
fraud_idx = np.where(test_ds.y == 1)[0]
ranked = fraud_idx[np.argsort(-probs[fraud_idx])]
print(f"{len(fraud_idx)} fraud sequences in test; showing the top {min(4, len(ranked))} by model confidence")

## 3. Per-sequence audit trail

For each flagged sequence: top panel = attention over the window (orange = padding); bottom = the monitored features. Attention should spike where behavior turns anomalous.

In [ ]:
feat_idx = {f: FEATURE_COLUMNS.index(f) for f in MONITORED_FEATURES}

def plot_sequence(i):
    a = attn[i]
    pad = test_ds.mask[i]
    fig, ax = plt.subplots(2, 1, figsize=(11, 5), sharex=True,
                           gridspec_kw={"height_ratios": [1, 1.3]})
    colors = ["orange" if p else "steelblue" for p in pad]
    ax[0].bar(range(SEQUENCE_LENGTH), a, color=colors)
    ax[0].set_ylabel("attention")
    ax[0].set_title(f"test seq #{i}  |  P(fraud)={probs[i]:.3f}  |  label={int(test_ds.y[i])}"
                    f"   (orange = padding)")
    for f, j in feat_idx.items():
        ax[1].plot(range(SEQUENCE_LENGTH), test_ds.X[i, :, j], marker=".", label=f)
    ax[1].set_xlabel("timestep (0 = oldest, 49 = most recent)")
    ax[1].set_ylabel("scaled value"); ax[1].legend(loc="upper left")
    # mark the peak-attention timestep
    peak = int(np.argmax(a))
    for axi in ax:
        axi.axvline(peak, color="red", ls="--", alpha=0.5)
    plt.tight_layout(); plt.show()
    print(f"peak attention at timestep {peak} (most-recent={SEQUENCE_LENGTH-1})")

for i in ranked[:4]:
    plot_sequence(int(i))

## 4. Where does attention concentrate, on average?

If the model learned the takeover pattern, fraud sequences should attend to the **recent** end of the window (where the burst happens), more so than legit sequences.

In [ ]:
def mean_attn(indices):
    A = attn[indices].copy()
    A[test_ds.mask[indices]] = np.nan       # ignore padding
    return np.nanmean(A, axis=0)

fraud_mean = mean_attn(np.where(test_ds.y == 1)[0])
legit_mean = mean_attn(np.where(test_ds.y == 0)[0])

plt.figure(figsize=(11, 4))
plt.plot(fraud_mean, label="fraud sequences", color="crimson")
plt.plot(legit_mean, label="legit sequences", color="steelblue")
plt.xlabel("timestep (0 = oldest, 49 = most recent)")
plt.ylabel("mean attention (padding ignored)")
plt.title("Average attention by position"); plt.legend(); plt.show()

recent = lambda v: np.nansum(v[-10:]) / np.nansum(v)
print(f"share of attention on last 10 steps — fraud: {recent(fraud_mean):.1%}  legit: {recent(legit_mean):.1%}")

## 5. Findings

> Fill in after running (numbers vary by seed / on real data).

- Flagged sequences put **most of their attention on the recent timesteps**,
  consistent with account-takeover bursts appearing late in the history.
- The peak-attention step typically coincides with a spike in `amount_zscore`
  and `txn_velocity_1h` and a `new_device_flag` — i.e. the model's explanation
  matches the injected fraud pattern.
- This is the artifact to show a fraud analyst: *"flagged because of these
  specific recent events,"* not an opaque score.

**Next:** repeat on the real IEEE-CIS data and spot-check that attention lands
on plausible transactions for true positives, and investigate false positives
where attention concentrates on benign events.